In [104]:
import torch
import torch.nn as nn
import math

In [23]:
# Some parameters for MHA
embed_dimension = 10
n_samples = 1
n_particles = 3
num_heads = 1

In [211]:
# Some dummy inputs
x = torch.rand(n_particles, n_samples, embed_dimension)
print(x)
print(x.shape)
attention_mask = torch.tensor([[[0,1,1], [1,0,1], [1,1,0]]], dtype=torch.float)
key_mask = torch.tensor([[0,1,0]], dtype=torch.bool)
print(attention_mask)
print(key_mask)

tensor([[[0.4336, 0.6295, 0.0235, 0.1833, 0.6650, 0.3092, 0.6450, 0.1575,
          0.2662, 0.4281]],

        [[0.1686, 0.5268, 0.4932, 0.3199, 0.0367, 0.8508, 0.2361, 0.1900,
          0.0150, 0.3126]],

        [[0.9978, 0.5091, 0.0663, 0.1301, 0.5629, 0.1235, 0.8753, 0.8101,
          0.5910, 0.7076]]])
torch.Size([3, 1, 10])
tensor([[[0., 1., 1.],
         [1., 0., 1.],
         [1., 1., 0.]]])
tensor([[False,  True, False]])


In [212]:
# Initialize a MHA layer
mha = nn.MultiheadAttention(embed_dimension, num_heads=num_heads, bias=False)

In [213]:
# Forward pass of MHA
y, weights = mha.forward(x, x, x, average_attn_weights=False, key_padding_mask=key_mask)
print(y)
print(y.shape)
print(weights)

tensor([[[-0.0442, -0.2926, -0.0262, -0.0033, -0.0799, -0.2261, -0.2594,
           0.1000, -0.1343, -0.4181]],

        [[-0.0460, -0.2904, -0.0265, -0.0031, -0.0788, -0.2254, -0.2555,
           0.0991, -0.1356, -0.4133]],

        [[-0.0454, -0.2911, -0.0264, -0.0032, -0.0792, -0.2256, -0.2568,
           0.0994, -0.1351, -0.4150]]], grad_fn=<ViewBackward0>)
torch.Size([3, 1, 10])
tensor([[[[0.5109, 0.0000, 0.4891],
          [0.5251, 0.0000, 0.4749],
          [0.5201, 0.0000, 0.4799]]]], grad_fn=<ViewBackward0>)


In [210]:
x_new = x
# x_new[0,0,:] = torch.zeros_like(x_new[0,0,:])
# print(x_new)
y_mod, weights = mha.forward(x_new, x_new, x_new, average_attn_weights=False, key_padding_mask=key_mask, attn_mask=attention_mask)
print(y_mod)
print(y_mod.shape)
print(weights)

tensor([[[-0.4250,  0.0398, -0.3048,  0.2119, -0.0194, -0.0331, -0.0089,
           0.1299,  0.0761,  0.2241]],

        [[-0.4236,  0.0664, -0.3401,  0.1997, -0.0148, -0.0499,  0.0125,
           0.1402,  0.0902,  0.2307]],

        [[-0.4264,  0.0141, -0.2706,  0.2237, -0.0238, -0.0169, -0.0296,
           0.1199,  0.0625,  0.2176]]], grad_fn=<ViewBackward0>)
torch.Size([3, 1, 10])
tensor([[[[0.0000, 0.4950, 0.5050],
          [0.0000, 0.2590, 0.7410],
          [0.0000, 0.7234, 0.2766]]]], grad_fn=<ViewBackward0>)


/Users/kevingreif/anaconda3/envs/zjets_ur/lib/python3.11/site-packages/torch/nn/functional.py:5076: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [163]:
# Implement using my own attention using the initialized weights
ipw = mha.in_proj_weight
q_w = x[:,0,:] @ ipw[:10,:].t()
k_w = x[:,0,:] @ ipw[10:20,:].t()
v_w = x[:,0,:] @ ipw[20:30,:].t()
# print(q_w.shape)
# print(k_w.shape)
print(v_w.shape)
softmax_output = torch.softmax((q_w @ k_w.t()) / 3, dim=1)
print(softmax_output)
attention = softmax_output @ v_w
# print(attention)
final_output = attention @ mha.out_proj.weight.t()
print(final_output)

torch.Size([3, 10])
tensor([[0.3333, 0.3333, 0.3333],
        [0.2928, 0.3183, 0.3890],
        [0.2940, 0.3373, 0.3687]], grad_fn=<SoftmaxBackward0>)
tensor([[ 0.2322,  0.0331, -0.0212,  0.0588, -0.0963,  0.1243,  0.0158, -0.2528,
         -0.1204,  0.0526],
        [ 0.2586,  0.0329, -0.0236,  0.0577, -0.1037,  0.1296,  0.0087, -0.2713,
         -0.1316,  0.0554],
        [ 0.2514,  0.0341, -0.0229,  0.0602, -0.1027,  0.1306,  0.0131, -0.2692,
         -0.1292,  0.0555]], grad_fn=<MmBackward0>)
